### Text preprocess

In [3]:
import nltk
from nltk import tokenize
import numpy as np 
from string import punctuation
import unidecode
import pandas as pd
#stemmer = nltk.RSLPStemmer()


def proccess_text(tweets):
    
    # Removing links, mentions and hashtags
    tweets['processed_text'] = tweets.text.str.replace(r'(http\S+)', '',regex=True) \
                                          .str.replace(r'@[\w]*', '',regex=True) \
                                          .str.replace(r'#[\w]*','',regex=True) 
    print('[ok] - Removing links.')
    print('[ok] - Removing mentions.')
    print('[ok] - Removing hashtags.')

    textWords = ' '.join([text for text in tweets.processed_text])

    # Removing accent
    textWords = [unidecode.unidecode(text) for text in tweets.processed_text ]    
    print('[ok] - Removing accent.')
    
    # Creating a list of words and characters (stopwords) to be removed from the text
    # stopWords = nltk.corpus.stopwords.words("portuguese")    
    print('[ok] - Creating a list of words and characters (stopwords) to be removed from the text.')
    
    
    # Separating punctuation from words
    punctSeparator = tokenize.WordPunctTokenizer()
    punctuationList = list()
    for punct in punctuation:
        punctuationList.append(punct)
        
    #stopWords =   punctuationList + stopWords    
    stopWords =   punctuationList
    #print('[ok] - Separating punctuation from words.')


    # Iterating over the text and removing stop words 
    trasnformedText = list()    
    for text in textWords:
        newText = list()   
        text = text.lower()
        textWords = punctSeparator.tokenize(text)
        for words in textWords:
             if words not in stopWords:
                #newText.append(stemmer.stem(words))
                newText.append(words)
        trasnformedText.append(' '.join(newText))
    tweets.processed_text = trasnformedText
    print('[ok] - Removing punctuation and set text to lowecase.')
   
    # Removing all non-text characters
    tweets.processed_text = tweets['processed_text'].str.replace(r"[^a-zA-Z#]", " ", regex=True)                                                         
    print('[ok] - Removing all non-text characters.')
   
    trasnformedText = list()
    for phrase in tweets.processed_text:
        newPhrase = list()   
        newPhrase.append(' '.join(phrase.split()))
        for words in newPhrase:
            trasnformedText.append(''.join(newPhrase))
    tweets.processed_text = trasnformedText
    
    # Removing tweets with less than three terms
    index=[x for x in tweets.index if tweets.processed_text[x].count(' ') < 3]
    tweets = tweets.drop(index)
    print('[ok] - Removing tweets with less than three terms.')

    # Removing empty lines
    removeEmpty  = tweets.processed_text != ' '
    tweets = tweets[removeEmpty]
    print('[ok] - Removing empty lines.')

    tweets.reset_index(inplace=True)
    #tweets = {'created_at': tweets.created_at, 'id':tweets.id,'author_id':tweets.author_id,'in_reply_to_user_id':tweets.in_reply_to_user_id, 'text': tweets.processed_text}
    tweets = {'text': tweets.processed_text}
    tweets = pd.DataFrame(tweets)
    #tweets = tweets.sort_values(['created_at']).reset_index().drop(columns=["index"])
    #tweets = tweets.reset_index().drop(columns=["index"])
    
    return tweets

### Criando dataset com os dados coletados usando as hashtags #DonaldTrump e #trump2016 

- Essas hashtags foram usadas para construir o corpus textual do evento semeval2016

In [ ]:
import pandas as pd 

trump2016 = pd.read_csv('../data/raw/trump2016.csv')
trump2016 = trump2016[['id','author_id','text']]
trump2016 = trump2016.dropna()



donaldtrump = pd.read_csv(open('../data/raw/donaldtrump.csv','rU'), encoding='utf-8')
donaldtrump = donaldtrump[['id','author_id','text']]
donaldtrump = donaldtrump.dropna()


trump = pd.concat([trump2016,donaldtrump])
trump = trump.reset_index().drop(columns=['index'])
trump.to_csv('../data/raw/trump_raw.csv', index=False)
len(trump)

### Dados originais disponibilizados via tweet ids

- A maioria dos tweets correspondente aos ids disponibilizados no site do semeval 2016 não existem mais
- Minha orientadora conseguiu os dados originais através de um ex-aluno

In [ ]:
marcelo = pd.read_csv('../data/raw/marcelo_raw.csv')

trump = trump[trump.id.isin(marcelo.id)]
len(trump)

### Mergeando os dados originais com os coletados

In [ ]:
trump_marcelo_concact = pd.concat([trump,marcelo])
trump_marcelo_concact = trump_marcelo_concact.reset_index().drop(columns=['index'])
len(trump_marcelo_concact)
trump_marcelo_concact.to_csv('../data/raw/trump_marcelo_concact.csv', index=False)

### Pre-processando os dados mergeados

In [ ]:
tweets = proccess_text(trump_marcelo_concact)
tweets.to_csv('../data/processed/trump_marcelo_processed.csv', index=False)

### Teste AWS

In [4]:
import pandas as pd

aws = pd.read_csv('../data/raw/aws_reviews.csv')
aws = aws.rename(columns={'Text':'text'})
aws['text'] = proccess_text(aws)
aws = aws.dropna().reset_index().drop(columns=['index'])

[ok] - Removing links.
[ok] - Removing mentions.
[ok] - Removing hashtags.
[ok] - Removing accent.
[ok] - Creating a list of words and characters (stopwords) to be removed from the text.
[ok] - Removing punctuation and set text to lowecase.
[ok] - Removing all non-text characters.
[ok] - Removing tweets with less than three terms.
[ok] - Removing empty lines.


In [8]:
def partition(x):
    if x < 3:
        return 1
    return 0

#changing reviews with score less than 3 to be positive
actualScore = aws['Score']
positiveNegative = actualScore.map(partition) 
aws['label'] = positiveNegative

In [14]:
positive = aws[aws.label == 0].sample(n=2000, random_state=44)
negative = aws[aws.label == 1].sample(n=2000, random_state=44)
query_set_aws = pd.concat([negative,positive])

In [ ]:
aws = aws[~aws.ProductId.isin(query_set_aws.ProductId)]

In [ ]:
query_set_aws = query_set_aws[['text', 'label']]
query_set_aws.to_csv('../data/query_set_aws.csv', index= False)
aws.to_csv('../data/processed/aws.csv', index=False)

### Dataset de validação

In [12]:
import pandas as pd
val = pd.DataFrame()
validation = pd.read_csv('../data/raw/query_set_gold.csv')
val = validation[['text','label']]
val['text'] = proccess_text(validation)
val = val.dropna().reset_index().drop(columns='index')
val.to_csv('../data/validation.csv',index=False)


[ok] - Removing links.
[ok] - Removing mentions.
[ok] - Removing hashtags.
[ok] - Removing accent.
[ok] - Creating a list of words and characters (stopwords) to be removed from the text.
[ok] - Removing punctuation and set text to lowecase.
[ok] - Removing all non-text characters.
[ok] - Removing tweets with less than three terms.
[ok] - Removing empty lines.


### Extract more frequent hashtags

In [ ]:
import re
import nltk
from nltk import tokenize

def count_hashtags(data_of_text):
    regex = re.compile(r'#[\w]*')
    #regex = re.compile(r'#(\w*[0-9a-zA-Z]+\w*[0-9a-zA-Z])')

    textWords = ' '.join([text for text in data_of_text])

    hashtags = regex.findall(textWords)

    hashtags = ' '.join([text for text in  hashtags])

    tokenizing = tokenize.WhitespaceTokenizer()
    tokenizedWords = tokenizing.tokenize(hashtags)
    frequency = nltk.FreqDist(tokenizedWords)
    df_frequency = pd.DataFrame({"Hashtag": list(frequency.keys()),
                                       "Frequency": list(frequency.values())})

    return df_frequency

In [ ]:
import nltk
from nltk import tokenize
import numpy as np 
from string import punctuation
import unidecode


    #tokens = tokenize.WordPunctTokenizer()
    #tokens = tokenize.WhitespaceTokenizer()
    #tokens = tokenize.MWETokenizer()

def proccess_text(data_of_text, tk):
    textWords = [unidecode.unidecode(text) for text in data_of_text]       
    
    
    punctuationList = list()
    for punct in punctuation:
        if punct != '#':
            punctuationList.append(punct)
    trasnformedText = list()
    
    personalList=["...","!#","@#","'#",".#","\"#","...#","(#","?#","!!"]  
    punctuationList = punctuationList  
    
    for text in textWords:
        newText = list()   
        text = text.lower()
        textWords = tk.tokenize(text)
        for words in textWords: 
             if words not in punctuationList:
                 newText.append(words)
        trasnformedText.append(' '.join(newText))
    return trasnformedText

In [ ]:
import pandas as pd

tweets = pd.read_csv('../data/raw/trump_marcelo_concact.csv')
tweets

In [ ]:
tweets['processed_text'] = proccess_text(tweets.text, tokenize.WhitespaceTokenizer())
tweets


In [ ]:
df = count_hashtags(tweets.processed_text)
df

In [ ]:
df.head()

In [ ]:
import tabloo

tabloo.show(df)